# GDP Preprocessing — Nigeria (NGA) 2024

Two extraction strategies from the Kummu v4 dataset:
- **Windowed** (`preprocess_kummu_gdp_tif_windowed`): memory-light, recommended for large global rasters
- **General extractor** (`extract_country_gdp`): auto-downloads boundary, clips, saves

In [4]:
import os, sys
ROOT_DIR = os.path.abspath(os.curdir)
sys.path.insert(0, ROOT_DIR)

from preprocess_kummu_gdp import download_file, preprocess_kummu_gdp_tif_windowed, KUMMU_V4_FILES
from extract_gdp import extract_country_gdp
import geopandas as gpd

gdp_path = os.path.join(ROOT_DIR, "GDP_data")
admin_path = os.path.join(ROOT_DIR, "admin")
os.makedirs(gdp_path, exist_ok=True)

In [5]:
# Download Kummu v4 30 arc-sec total GDP raster (1.4 GB - skips if already present)
tif_path = download_file(
    url=KUMMU_V4_FILES["rast_gdpTot_1990_2024_30arcsec.tif"],
    out_dir=gdp_path,
)
print("Source raster:", tif_path)

Already exists, skipping download: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_Nigeria\GDP_data\rast_gdpTot_1990_2024_30arcsec.tif
Source raster: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_Nigeria\GDP_data\rast_gdpTot_1990_2024_30arcsec.tif


## Strategy 1 - Windowed extraction (requires local boundary shapefile)

In [6]:
# Load Nigeria ADM2 boundary (downloads from geoBoundaries if not present)
from extract_gdp import _get_country_boundary

_shp = os.path.join(admin_path, "Nigeria_admin2.shp")
if not os.path.exists(_shp):
    os.makedirs(admin_path, exist_ok=True)
    nga_gdf = _get_country_boundary("NGA", adm_level="ADM2")
    nga_gdf.to_file(_shp)
    print(f"Downloaded and saved {len(nga_gdf)} LGAs to {_shp}")
else:
    nga_gdf = gpd.read_file(_shp)
    print(f"Loaded {len(nga_gdf)} LGAs from {_shp}")

da_windowed = preprocess_kummu_gdp_tif_windowed(
    tif_path=tif_path,
    year=2024,
    boundary_gdf=nga_gdf,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_NGA_windowed.tif"),
)
print("Total Nigeria GDP 2024 (windowed):", float(da_windowed.sum()))

Loaded NGA ADM2 boundary: 774 feature(s), CRS=EPSG:4326
Downloaded and saved 774 LGAs to C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_Nigeria\admin\Nigeria_admin2.shp
Window bounds (padded ±0.05°): (2.6426, 4.2202, 14.728, 13.9357)
No usable embedded band-year tags; assuming order: (1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024)
  -> VERIFY this matches github.com/mattikummu/griddedGDPpc before trusting results.
Reading window: Window(col_off=21918, row_off=9019, width=1450, height=1166) (band 8 -> year 2024)
Window shape: (1166, 1450) | valid pixels: 1604291 / 1690700
Post-clip shape: (1154, 1438) | valid: 1083139 / 1659452
Saved: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_Nigeria\GDP_data\GDP_PPP_2024_NGA_windowed.tif
Total Nigeria GDP 2024 (windowed): 1503178391552.0


## Strategy 2 - General extractor (auto-downloads boundary from geoBoundaries API)

In [7]:
da = extract_country_gdp(
    country="NGA",
    year=2024,
    gdp_path=gdp_path,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_NGA.tif"),
)
print("Total Nigeria GDP 2024 (general extractor):", float(da.sum()))

Loaded NGA ADM0 boundary: 1 feature(s), CRS=EPSG:4326
NGA (NGA) 2024: 730351 valid pixels, total GDP=1,503,452,594,176
Saved: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_Nigeria\GDP_data\GDP_PPP_2024_NGA.tif
Total Nigeria GDP 2024 (general extractor): 1503452594176.0
